# Ateneo+ Pipeline v2 — Fase 6: Fine-Tuning de BGE-M3 con MNRL (NVIDIA A100 40GB)

Este notebook ejecuta el re-entrenamiento del modelo de embeddings denso **`BAAI/bge-m3`** utilizando los datasets del Pipeline Científico de Ingesta v2 de **Ateneo+**.

### Especificaciones Técnicas y de Hardware:
* **Hardware Objetivo:** NVIDIA A100 (40 GB VRAM / ~83 GB RAM de sistema).
* **Precisión:** `bfloat16` nativo en Tensor Cores Ampere.
* **Función de Pérdida:** `MultipleNegativesRankingLoss` (MNRL) con $\tau = 0.02$ (Scale = 50.0).
* **Batch Size:** 32 tripletas efectivas (31 in-batch negatives por paso de optimización).
* **Datasets:** `retrieval_train.json` (1,178 tripletas) y `retrieval_val.json` (249 tripletas).
* **Tiempo estimado:** ~10 a 15 minutos para 5 épocas.

In [ ]:
# 1. Diagnóstico de Hardware NVIDIA A100
!nvidia-smi

import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"GPU Dispositivo : {gpu_name} ({vram_gb:.1f} GB VRAM)")
    print(f"Soporte BF16    : {bf16_ok} (Ampere Tensor Cores)")

In [ ]:
# 2. Instalación de dependencias de entrenamiento
!pip install -q "sentence-transformers>=3.0.0" "datasets>=2.18.0" "accelerate>=0.28.0"

In [ ]:
# 3. Preparación de Datasets y Entorno
import os, json, hashlib
from pathlib import Path

os.makedirs("datasets", exist_ok=True)
os.makedirs("models/ateneo-bge-m3-ecuador-v2", exist_ok=True)

print("Directorio de datasets preparado.")
print("Asegúrate de que 'retrieval_train.json' y 'retrieval_val.json' estén en la carpeta 'datasets/' o en la raíz.")

In [ ]:
# 4. Carga y Verificación Criptográfica de Datasets
train_path = Path("datasets/retrieval_train.json") if Path("datasets/retrieval_train.json").exists() else Path("retrieval_train.json")
val_path = Path("datasets/retrieval_val.json") if Path("datasets/retrieval_val.json").exists() else Path("retrieval_val.json")

with open(train_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open(val_path, "r", encoding="utf-8") as f:
    val_data = json.load(f)

train_hash = hashlib.sha256(train_path.read_bytes()).hexdigest()
val_hash = hashlib.sha256(val_path.read_bytes()).hexdigest()

print(f"Train samples: {len(train_data)} | SHA-256: {train_hash[:16]}...")
print(f"Val samples  : {len(val_data)}   | SHA-256: {val_hash[:16]}...")

In [ ]:
# 5. Construcción del Evaluador y Ejecución del Fine-Tuning con bfloat16
import time
from datasets import Dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.evaluation import InformationRetrievalEvaluator

# Formatear tripletas
train_triplets = [{"anchor": item["query"].strip(), "positive": item["pos"].strip(), "negative": item["neg"].strip()} for item in train_data]
train_dataset = Dataset.from_list(train_triplets)

# Construir IR Evaluator para Validación
corpus, queries, relevant_docs = {}, {}, {}
text_to_id = {}
for idx, item in enumerate(val_data):
    q_id = f"q_{idx:04d}"
    queries[q_id] = item["query"].strip()
    
    pos_text = item["pos"].strip()
    neg_text = item["neg"].strip()
    
    if pos_text not in text_to_id:
        p_id = f"doc_pos_{idx:04d}"
        text_to_id[pos_text] = p_id
        corpus[p_id] = pos_text
    p_id = text_to_id[pos_text]
    
    if neg_text not in text_to_id:
        n_id = f"doc_neg_{idx:04d}"
        text_to_id[neg_text] = n_id
        corpus[n_id] = neg_text
        
    relevant_docs[q_id] = {p_id}

evaluator = InformationRetrievalEvaluator(
    queries=queries, corpus=corpus, relevant_docs=relevant_docs,
    name="ateneo_val_evaluator", mrr_at_k=[1, 5, 10], ndcg_at_k=[1, 5, 10], accuracy_at_k=[1, 3, 5],
    show_progress_bar=True
)

# Cargar BGE-M3 base
model = SentenceTransformer("BAAI/bge-m3", device="cuda")
model.max_seq_length = 512

# Loss MNRL con escala 50.0 (tau = 0.02)
loss_fn = MultipleNegativesRankingLoss(model, scale=50.0)

# Argumentos optimizados para A100 40GB
training_args = SentenceTransformerTrainingArguments(
    output_dir="checkpoints_bge_m3",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.10,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    weight_decay=0.01,
    bf16=True, # bfloat16 nativo en la A100
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="ateneo_val_evaluator_cosine_mrr@5",
    greater_is_better=True,
    logging_steps=10,
    report_to="none",
    seed=42,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    loss=loss_fn,
    evaluator=evaluator,
)

print("=== INICIANDO ENTRENAMIENTO EN NVIDIA A100 ===")
t0 = time.time()
trainer.train()
print(f"Entrenamiento completado en {(time.time() - t0)/60.0:.2f} minutos.")

# Guardar modelo definitivo
final_path = "models/ateneo-bge-m3-ecuador-v2"
model.save_pretrained(final_path)
print(f"Modelo final exportado en: {final_path}")

In [ ]:
# 6. Empaquetado comprimido para integración en ChromaDB
!tar -czvf ateneo-bge-m3-ecuador-v2.tar.gz models/ateneo-bge-m3-ecuador-v2/
print("Archivo comprimido listo para descarga o despliegue:")
!ls -lh ateneo-bge-m3-ecuador-v2.tar.gz